# 08 M1: พยากรณ์ผู้โดยสารเข้าสถานีรายชั่วโมงล่วงหน้า 7 วัน

การใช้งาน: ทุกปลายสัปดาห์ฝ่ายวางแผนต้องกำหนดความยาวขบวน ความถี่ และพนักงานของสัปดาห์หน้า
- P50 = แผนหลัก
- P90 = ขบวน/ตู้สำรองสำหรับกรณีความต้องการสูง
- P10 = ช่วงที่ลดความถี่ได้อย่างปลอดภัย

พยากรณ์ออกตอนปลายสัปดาห์สำหรับทั้งสัปดาห์หน้า ข้อมูลที่ใช้ได้จึงเก่าอย่างน้อย 7 วันเสมอ
ทุก feature ของวัน t ต้องมาจากข้อมูลวันที่ไม่เกิน t - 7

notebook นี้แบ่งเป็น 3 ส่วน
1. สร้าง feature และตรวจ leakage (ส่วนนี้)
2. เทรนด้วย trial น้อยบน GPU เทียบกับ baseline
3. จูนเต็ม, quantile P10/P50/P90, SHAP และประเมินบน test

In [1]:
import sys
from pathlib import Path

if Path.cwd().name == "notebooks":
    ROOT = Path.cwd().parent
else:
    ROOT = Path.cwd()
sys.path.append(str(ROOT / "src"))

from datetime import date

import pandas as pd
import polars as pl

from bart import viz
from bart.config import FEATURES, MARTS, create_views, duck, fetch_value
from bart.perf import stage

viz.notebook_mode()

con = duck()
create_views(con)

## 1. สร้าง feature ด้วย Polars lazy

ทุกขั้นเป็น lazy query ยังไม่คำนวณจริงจนถึงตอน `collect()` Polars จะรวมทุกขั้นเป็นแผนเดียวแล้วรันขนานทุก core

lag ทำด้วยการ join ตามวันที่ (เลื่อนวันที่ของข้อมูลต้นทางไป +7k วัน แล้ว join กับวันเป้าหมาย) ไม่ใช้การเลื่อนแถว
ถ้ามีวันที่ขาดหายจะไม่หยิบค่าของวันผิดมาใช้

ค่าต้นทางของ lag ใช้เฉพาะวัน normal วันอื่น (strike, partial, no_data) ให้เป็นค่าว่าง

In [2]:
calendar = pl.scan_parquet(MARTS / "calendar.parquet")
station_hour = pl.scan_parquet(MARTS / "station_hour.parquet")
station_day = pl.scan_parquet(MARTS / "station_day.parquet")
system_hour = pl.scan_parquet(MARTS / "system_hour.parquet")
stations = pl.scan_parquet(MARTS / "stations.parquet")

day_status = calendar.select("date", "day_status")
is_normal = pl.col("day_status") == "normal"


def only_normal(column):
    return pl.when(is_normal).then(pl.col(column).cast(pl.Float64)).otherwise(None)

### 1.1 lag ชั่วโมงเดียวกัน 1, 2, 3, 4, 52 สัปดาห์

In [3]:
hourly_source = (
    station_hour
    .join(day_status, on="date", how="left")
    .select("station", "date", "hour", only_normal("entries").alias("value"))
)


def lag_frame(weeks):
    days = 7 * weeks
    return hourly_source.select(
        pl.col("station"),
        (pl.col("date") + pl.duration(days=days)).alias("date"),
        pl.col("hour"),
        pl.col("value").alias(f"lag_{weeks}w"),
    )


lag_weeks = [1, 2, 3, 4, 52]

### 1.2 ระดับล่าสุดของสถานีและของทั้งระบบ

- station_mean_7d / 28d: entries เฉลี่ยต่อวันของสถานี ช่วง 7 และ 28 วันที่จบที่ t - 7
- station_trend = 7d / 28d (มากกว่า 1 แปลว่ากำลังเพิ่ม)
- system_lag_1w: ทริปทั้งระบบชั่วโมงเดียวกันเมื่อ 7 วันก่อน
- system_mean_7d: ทริปทั้งระบบเฉลี่ยต่อวัน ช่วง 7 วันที่จบที่ t - 7

station_day มีทุกวันตั้งแต่วันเปิดสถานี (วันที่ไม่มีข้อมูลเป็นค่าว่าง) จึงใช้ rolling ตามลำดับแถวได้

In [4]:
station_level = (
    station_day
    .join(day_status, on="date", how="left")
    .select("station", "date", only_normal("entries").alias("value"))
    .sort("station", "date")
    .with_columns(
        pl.col("value").rolling_mean(window_size=7, min_samples=4).over("station").alias("station_mean_7d"),
        pl.col("value").rolling_mean(window_size=28, min_samples=14).over("station").alias("station_mean_28d"),
    )
    .select(
        pl.col("station"),
        (pl.col("date") + pl.duration(days=7)).alias("date"),
        "station_mean_7d",
        "station_mean_28d",
    )
)

system_hour_lag = (
    system_hour
    .join(day_status, on="date", how="left")
    .select(
        (pl.col("date") + pl.duration(days=7)).alias("date"),
        pl.col("hour"),
        only_normal("trips").alias("system_lag_1w"),
    )
)

system_level = (
    calendar
    .sort("date")
    .select("date", only_normal("day_trips").alias("value"))
    .with_columns(pl.col("value").rolling_mean(window_size=7, min_samples=4).alias("system_mean_7d"))
    .select((pl.col("date") + pl.duration(days=7)).alias("date"), "system_mean_7d")
)

### 1.3 ปฏิทิน

- day_of_week, month, is_holiday, holiday_name
- days_to_holiday / days_since_holiday: จำนวนวันถึงวันหยุดถัดไป และหลังวันหยุดก่อนหน้า (ตัดที่ 30 วัน)
- lag_1w_was_holiday: วันเดียวกันเมื่อสัปดาห์ก่อนเป็นวันหยุดหรือไม่ (baseline พลาดหนักในกรณีนี้)
- is_pride_day: อาทิตย์สุดท้ายของเดือนมิถุนายน (EDA ส่วนที่ 2)
- covid_period: pre / break / post

In [5]:
holiday_dates = (
    calendar
    .filter(pl.col("is_holiday"))
    .select(pl.col("date").alias("holiday_date"))
    .sort("holiday_date")
)

calendar_features = (
    calendar
    .sort("date")
    .join_asof(holiday_dates, left_on="date", right_on="holiday_date", strategy="forward")
    .rename({"holiday_date": "next_holiday"})
    .join_asof(holiday_dates, left_on="date", right_on="holiday_date", strategy="backward")
    .rename({"holiday_date": "previous_holiday"})
    .select(
        "date",
        "day_status",
        "day_of_week",
        pl.col("date").dt.month().alias("month"),
        "is_holiday",
        pl.col("holiday_name").fill_null("none"),
        (pl.col("next_holiday") - pl.col("date")).dt.total_days().clip(0, 30).fill_null(30).alias("days_to_holiday"),
        (pl.col("date") - pl.col("previous_holiday")).dt.total_days().clip(0, 30).fill_null(30).alias("days_since_holiday"),
        (
            (pl.col("date").dt.month() == 6)
            & (pl.col("day_of_week") == 7)
            & (pl.col("date").dt.day() >= 24)
        ).alias("is_pride_day"),
        "covid_period",
    )
)

previous_week_holiday = calendar.select(
    (pl.col("date") + pl.duration(days=7)).alias("date"),
    pl.col("is_holiday").alias("lag_1w_was_holiday"),
)

### 1.4 รวมทุกอย่าง

target = entries ของวัน normal เท่านั้น
split ตามเวลา
- train: ถึง 2022-06-30 ไม่รวมช่วง break (2020-03-09 ถึง 2020-05-31)
- valid: 2022-07-01 ถึง 2022-12-31 ใช้จูนและ early stopping
- test: 2023-01-01 ถึง 2023-05-02 ใช้ครั้งเดียวตอนจบ

In [6]:
def split_label():
    in_break = pl.col("date").is_between(date(2020, 3, 9), date(2020, 5, 31))
    return (
        pl.when(pl.col("date") >= date(2023, 1, 1)).then(pl.lit("test"))
        .when(pl.col("date") >= date(2022, 7, 1)).then(pl.lit("valid"))
        .when(in_break).then(pl.lit("excluded"))
        .otherwise(pl.lit("train"))
    )


lag_columns = []
for weeks in lag_weeks:
    lag_columns.append(f"lag_{weeks}w")
recent_lags = ["lag_1w", "lag_2w", "lag_3w", "lag_4w"]

features = (
    station_hour
    .join(calendar_features, on="date", how="left")
    .filter(pl.col("day_status") == "normal")
    .join(stations.select("station", "corridor", "opening_date"), on="station", how="left")
    .with_columns(
        pl.col("entries").cast(pl.Float64).alias("target"),
        (pl.col("date") - pl.col("opening_date")).dt.total_days().alias("days_since_open"),
    )
)

for weeks in lag_weeks:
    features = features.join(lag_frame(weeks), on=["station", "date", "hour"], how="left")

features = (
    features
    .join(previous_week_holiday, on="date", how="left")
    .join(station_level, on=["station", "date"], how="left")
    .join(system_hour_lag, on=["date", "hour"], how="left")
    .join(system_level, on="date", how="left")
    .with_columns(
        pl.mean_horizontal(recent_lags).alias("lag_4w_mean"),
        pl.concat_list(recent_lags).list.median().alias("lag_4w_median"),
        pl.concat_list(recent_lags).list.std().alias("lag_4w_std"),
        pl.min_horizontal(recent_lags).alias("lag_4w_min"),
        pl.max_horizontal(recent_lags).alias("lag_4w_max"),
        (pl.col("station_mean_7d") / pl.col("station_mean_28d")).alias("station_trend"),
        split_label().alias("split"),
    )
    .drop("day_status", "opening_date", "entries", "exits")
    .sort("date", "hour", "station")
)

plan_lines = features.explain().splitlines()
print(len(plan_lines), "lines in the optimized plan, first 20:")
for line in plan_lines[:20]:
    print(line[:160])

152 lines in the optimized plan, first 20:
SORT BY [col("date"), col("hour"), col("station")]
  simple π 31/31 ["station", "date", "hour", ... 28 other columns]
     WITH_COLUMNS:
     [col("lag_1w").mean_horizontal([col("lag_2w"), col("lag_3w"), col("lag_4w")]).alias("lag_4w_mean"), col("__POLARS_CSER_0x100000036").alias("lag_1w").list.m
       WITH_COLUMNS:
       [col("lag_1w").list.concat([col("lag_2w"), col("lag_3w"), col("lag_4w")]).alias("__POLARS_CSER_0x100000036")] 
        LEFT JOIN:
        LEFT PLAN ON: [col("date")]
          LEFT JOIN:
          LEFT PLAN ON: [col("date"), col("hour")]
            LEFT JOIN:
            LEFT PLAN ON: [col("station"), col("date")]
              LEFT JOIN:
              LEFT PLAN ON: [col("date")]
                LEFT JOIN:
                LEFT PLAN ON: [col("station"), col("date"), col("hour")]
                  LEFT JOIN:
                  LEFT PLAN ON: [col("station"), col("date"), col("hour")]
                    LEFT JOIN:
            

In [7]:
with stage("08.build_features"):
    feature_table = features.collect()

print(f"{feature_table.height:,} rows x {feature_table.width} columns")
print(f"memory: {feature_table.estimated_size('mb'):.0f} MB")

feature_path = FEATURES / "m1_features.parquet"
with stage("08.write_features"):
    feature_table.write_parquet(feature_path, compression="zstd", compression_level=6)

print(f"written {feature_path.name}: {feature_path.stat().st_size / 1e6:.1f} MB")

[perf] 08.build_features: 2.10s | 16.17 cores | peak RAM 3.342 GB | CPU 81.0%
5,000,232 rows x 31 columns
memory: 943 MB


[perf] 08.write_features: 0.69s | 18.63 cores | peak RAM 3.388 GB | CPU 95.4%
written m1_features.parquet: 99.5 MB


In [8]:
feature_table.head(5).to_pandas()

,station,date,hour,day_of_week,month,is_holiday,holiday_name,days_to_holiday,days_since_holiday,is_pride_day,...,station_mean_28d,system_lag_1w,system_mean_7d,lag_4w_mean,lag_4w_median,lag_4w_std,lag_4w_min,lag_4w_max,station_trend,split
0,12TH,2011-01-01,0,6,1,True,New Year's Day,0,0,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,train
1,16TH,2011-01-01,0,6,1,True,New Year's Day,0,0,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,train
2,19TH,2011-01-01,0,6,1,True,New Year's Day,0,0,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,train
3,24TH,2011-01-01,0,6,1,True,New Year's Day,0,0,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,train
4,ASHB,2011-01-01,0,6,1,True,New Year's Day,0,0,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,train


## 2. สรุป feature

จำนวนแถวต่อ split และสัดส่วนค่าว่างของแต่ละ feature
ค่าว่างใน lag เกิดจาก ช่วงแรกของข้อมูล (lag 52 สัปดาห์ต้องรอ 1 ปี), สถานีที่เพิ่งเปิด และวันต้นทางที่ไม่ใช่ normal
XGBoost รับค่าว่างได้โดยตรง จึงไม่ต้องเติม

In [9]:
split_summary = (
    feature_table
    .group_by("split")
    .agg(
        pl.len().alias("rows"),
        pl.col("date").min().alias("first_date"),
        pl.col("date").max().alias("last_date"),
        pl.col("target").sum().alias("total_entries"),
    )
    .sort("first_date")
)

split_summary.to_pandas()

,split,rows,first_date,last_date,total_entries
0,train,4537416,2011-01-01,2022-06-30,1.152379e+09
1,excluded,95616,2020-03-09,2020-05-31,3.366113e+06
2,valid,220800,2022-07-01,2022-12-31,2.300418e+07
3,test,146400,2023-01-01,2023-05-02,1.545162e+07


In [10]:
feature_columns = [
    "station", "corridor", "hour", "day_of_week", "month", "is_holiday", "holiday_name",
    "days_to_holiday", "days_since_holiday", "is_pride_day", "covid_period", "days_since_open",
    "lag_1w", "lag_2w", "lag_3w", "lag_4w", "lag_52w",
    "lag_4w_mean", "lag_4w_median", "lag_4w_std", "lag_4w_min", "lag_4w_max",
    "lag_1w_was_holiday", "station_mean_7d", "station_mean_28d", "station_trend",
    "system_lag_1w", "system_mean_7d",
]

null_rows = []
train_rows = feature_table.filter(pl.col("split") == "train")
valid_rows = feature_table.filter(pl.col("split") == "valid")
for column in feature_columns:
    null_rows.append({
        "feature": column,
        "dtype": str(feature_table.schema[column]),
        "null_pct_train": round(100 * train_rows[column].null_count() / train_rows.height, 2),
        "null_pct_valid": round(100 * valid_rows[column].null_count() / valid_rows.height, 2),
    })

print(len(feature_columns), "features")
pd.DataFrame(null_rows)

28 features


,feature,dtype,null_pct_train,null_pct_valid
0,station,String,0.00,0.0
1,corridor,String,0.00,0.0
2,hour,UInt8,0.00,0.0
3,day_of_week,Int64,0.00,0.0
4,month,Int8,0.00,0.0
5,is_holiday,Boolean,0.00,0.0
6,holiday_name,String,0.00,0.0
7,days_to_holiday,Int64,0.00,0.0
8,days_since_holiday,Int64,0.00,0.0
9,is_pride_day,Boolean,0.00,0.0


## 3. ตรวจ leakage

### 3.1 สุ่มแถวมาคำนวณ lag ใหม่จาก station_hour ด้วย DuckDB
ใช้คนละเครื่องมือกับตอนสร้าง (Polars) ถ้าตรงกันทุกค่า แปลว่า lag ถูกต้องและมาจากวันที่ถูก

In [11]:
def values_match(source_value, feature_value):
    if source_value is None and feature_value is None:
        return True
    if source_value is None or feature_value is None:
        return False
    return float(source_value) == feature_value


sample = feature_table.filter(pl.col("lag_52w").is_not_null()).sample(n=20, seed=7)

mismatches = 0
checked_rows = []
for row in sample.iter_rows(named=True):
    for weeks in lag_weeks:
        source_value = fetch_value(con, f"""
            SELECT CASE WHEN c.day_status = 'normal' THEN h.entries END
            FROM station_hour AS h
            JOIN calendar AS c ON c.date = h.date
            WHERE h.station = '{row["station"]}'
              AND h.hour = {row["hour"]}
              AND h.date = DATE '{row["date"]}' - {7 * weeks}
        """)
        feature_value = row[f"lag_{weeks}w"]
        if not values_match(source_value, feature_value):
            mismatches += 1
    checked_rows.append({
        "station": row["station"], "date": row["date"], "hour": row["hour"],
        "target": row["target"], "lag_1w": row["lag_1w"], "lag_52w": row["lag_52w"],
    })

print("values checked:", len(checked_rows) * len(lag_weeks))
print("mismatches    :", mismatches)
assert mismatches == 0

pd.DataFrame(checked_rows).head(10)

values checked: 100
mismatches    : 0


,station,date,hour,target,lag_1w,lag_52w
0,COLS,2012-08-26,13,249.0,258.0,190.0
1,12TH,2014-01-25,10,330.0,256.0,256.0
2,SBRN,2020-05-07,16,29.0,17.0,177.0
3,DBRK,2017-01-27,6,106.0,104.0,115.0
4,16TH,2022-12-16,6,85.0,99.0,87.0
5,WCRK,2017-07-08,8,79.0,79.0,127.0
6,RICH,2020-06-01,16,40.0,18.0,158.0
7,12TH,2015-12-14,5,84.0,70.0,65.0
8,RICH,2023-02-24,23,10.0,7.0,4.0
9,WCRK,2012-11-15,17,522.0,521.0,470.0


### 3.2 วันต้นทางที่ไม่ใช่ normal ต้องได้ค่าว่าง ไม่ใช่ 0

เช่น target วันที่ 2013-07-08 ซึ่ง 7 วันก่อนคือวัน strike (2013-07-01) lag_1w ต้องเป็นค่าว่าง

In [12]:
non_normal_days = (
    calendar
    .filter(pl.col("day_status") != "normal")
    .select((pl.col("date") + pl.duration(days=7)).alias("date"))
    .collect()
)

after_non_normal = feature_table.join(non_normal_days, on="date", how="inner")
filled_by_mistake = after_non_normal.filter(pl.col("lag_1w").is_not_null()).height

print("rows whose lag_1w source day is not normal:", after_non_normal.height)
print("of those, lag_1w not null (should be 0):  ", filled_by_mistake)
assert after_non_normal.height > 0
assert filled_by_mistake == 0

rows whose lag_1w source day is not normal: 20544
of those, lag_1w not null (should be 0):   0


### 3.3 แถวแรกของสถานีใหม่ต้องไม่มี lag

ช่วง 7 วันแรกหลังเปิดสถานี ยังไม่มีข้อมูลเมื่อ 7 วันก่อน lag_1w ต้องเป็นค่าว่างทั้งหมด

In [13]:
first_week = feature_table.filter(pl.col("days_since_open").is_between(0, 6) & (pl.col("date") > date(2011, 1, 7)))
first_week_with_lag = first_week.filter(pl.col("lag_1w").is_not_null()).height

print("rows in the first week after a station opened:", first_week.height)
print("of those, lag_1w not null (should be 0):      ", first_week_with_lag)
assert first_week.height > 0
assert first_week_with_lag == 0

rows in the first week after a station opened: 1176
of those, lag_1w not null (should be 0):       0


### 3.4 baseline จาก feature ต้องได้ WAPE เท่ากับใน EDA

EDA (ส่วนที่ 5) คำนวณ baseline ในช่วง valid ด้วย SQL ได้ WAPE ค่าเฉลี่ย 4 สัปดาห์ 17.5% และสัปดาห์ก่อน 20.2%
ถ้าคำนวณจาก feature ชุดนี้แล้วได้ค่าเดียวกัน แปลว่า feature สร้างถูก

In [14]:
def wape(frame, prediction):
    rows = frame.filter(pl.col(prediction).is_not_null())
    error = float((rows["target"] - rows[prediction]).abs().sum())
    total = float(rows["target"].sum())
    return 100 * error / total


baseline_4w = wape(valid_rows, "lag_4w_mean")
baseline_1w = wape(valid_rows, "lag_1w")

print(f"valid WAPE, mean of 4 weeks: {baseline_4w:.2f}%  (EDA: 17.5%)")
print(f"valid WAPE, last week      : {baseline_1w:.2f}%  (EDA: 20.2%)")

assert abs(baseline_4w - 17.5) < 0.05
assert abs(baseline_1w - 20.2) < 0.05

valid WAPE, mean of 4 weeks: 17.54%  (EDA: 17.5%)
valid WAPE, last week      : 20.17%  (EDA: 20.2%)


## สรุปส่วนที่ 1

- สร้าง feature 28 ตัว ให้ทุกแถวของสถานี-ชั่วโมงในวัน normal ได้ 5,000,232 แถว ด้วย Polars lazy ใช้เวลา ~2 วินาที (RAM สูงสุด ~3 GB) เขียนเป็น Parquet ~100 MB
- train 4,537,416 แถว / valid 220,800 / test 146,400 / excluded (ช่วง break) 95,616
- ค่าว่างใน train น้อยมาก ยกเว้น lag_52w (~10% เพราะปีแรกของข้อมูลและสถานีใหม่) ส่วน valid ไม่มีค่าว่างเลย
- ตรวจ leakage ผ่านทั้ง 4 ข้อ
    - lag 100 ค่าที่สุ่มมาตรงกับค่าที่คำนวณใหม่ด้วย DuckDB ทุกค่า
    - วันต้นทางที่ไม่ใช่ normal ได้ค่าว่างทั้งหมด (20,544 แถว)
    - สัปดาห์แรกหลังเปิดสถานีไม่มี lag
    - baseline จาก feature ได้ WAPE 17.54% และ 20.17% ตรงกับ EDA

ต่อไป: ส่วนที่ 2 เทรนด้วย trial น้อยบน GPU

ต่อไป: ส่วนที่ 2 เทรนด้วย trial น้อยบน GPU

# ส่วนที่ 2: เทรนด้วย trial น้อยบน GPU

เป้าหมาย: ทดสอบว่า pipeline ครบ (ข้อมูล → GPU → ประเมินผล) ได้ผลเบื้องต้นเทียบกับ baseline และวัดเวลาต่อ trial ไว้ประมาณรอบจูนเต็ม
ส่วนนี้ใช้แค่ train กับ valid ไม่แตะ test

## 4. เตรียมข้อมูลเข้า XGBoost

- คอลัมน์ข้อความใช้ categorical ของ XGBoost โดยตรง (ไม่ทำ one-hot) และกำหนดรายการ category จากข้อมูลทั้งหมด ให้ train / valid / test ใช้รหัสเดียวกัน
- คอลัมน์ boolean แปลงเป็น 0/1 (lag_1w_was_holiday มีค่าว่างได้)
- ข้อมูลอยู่บน GPU ในรูป QuantileDMatrix (แบ่งค่าเป็น bin ล่วงหน้า ใช้ VRAM น้อยกว่า DMatrix ปกติ)

In [15]:
import time

import numpy as np
import optuna
import plotly.graph_objects as go
import xgboost as xgb

categorical_columns = ["station", "corridor", "holiday_name", "covid_period"]
boolean_columns = ["is_holiday", "is_pride_day", "lag_1w_was_holiday"]

categories = {}
for column in categorical_columns:
    values = feature_table[column].unique().sort().to_list()
    categories[column] = values


def to_model_input(frame):
    table = frame.select(feature_columns).to_pandas()
    for column in categorical_columns:
        table[column] = pd.Categorical(table[column], categories=categories[column])
    for column in boolean_columns:
        table[column] = table[column].astype("float32")
    return table


def is_workday():
    return (pl.col("day_of_week") <= 5) & ~pl.col("is_holiday")


train_frame = feature_table.filter(pl.col("split") == "train")
valid_frame = feature_table.filter(pl.col("split") == "valid").with_columns(is_workday().alias("is_workday"))

with stage("08.prepare_inputs"):
    X_valid = to_model_input(valid_frame)
    y_valid = valid_frame["target"].to_numpy()

print(f"train rows: {train_frame.height:,}")
print(f"valid rows: {valid_frame.height:,}")
print(f"features  : {len(feature_columns)}")

[perf] 08.prepare_inputs: 0.05s | 2.06 cores | peak RAM 2.014 GB
train rows: 4,537,416
valid rows: 220,800
features  : 28


## 5. ข้อมูล train ตั้งแต่ปีไหน

ระดับผู้โดยสารก่อน COVID ต่างจากตอนนี้มาก จึงลอง 3 แบบด้วย parameter ชุดเดียวกัน
- all: ทั้งหมดตั้งแต่ปี 2011
- weighted: ทั้งหมดตั้งแต่ปี 2011 แต่ข้อมูลเก่ามีน้ำหนักน้อยลงครึ่งหนึ่งทุก 2 ปี (half-life 2 ปี นับจาก 2022-07-01)
- post_covid: เฉพาะ มิ.ย. 2020 ถึง มิ.ย. 2022

ใช้ objective Tweedie (power 1.4 ตาม EDA) และ early stopping บน MAE ของ valid
(MAE กับ WAPE เรียงลำดับโมเดลเหมือนกัน เพราะ WAPE = ผลรวม error / ผลรวมค่าจริง บนชุดข้อมูลเดียวกัน)

In [16]:
valid_start = date(2022, 7, 1)


def training_rows(variant):
    if variant == "post_covid":
        return train_frame.filter(pl.col("date") >= date(2020, 6, 1))
    return train_frame


def recency_weights(frame, half_life_years):
    days_back = (pl.lit(valid_start) - pl.col("date")).dt.total_days()
    weights = frame.select((0.5 ** (days_back / 365.25 / half_life_years)).alias("weight"))
    return weights["weight"].to_numpy()


def build_matrices(variant, half_life_years=2.0):
    rows = training_rows(variant)
    X_train = to_model_input(rows)
    y_train = rows["target"].to_numpy()
    if variant == "weighted":
        weight = recency_weights(rows, half_life_years)
    else:
        weight = None
    train_matrix = xgb.QuantileDMatrix(X_train, y_train, weight=weight, enable_categorical=True, max_bin=256)
    valid_matrix = xgb.QuantileDMatrix(X_valid, y_valid, ref=train_matrix, enable_categorical=True, max_bin=256)
    return train_matrix, valid_matrix, rows.height


base_params = {
    "device": "cuda",
    "tree_method": "hist",
    "objective": "reg:tweedie",
    "tweedie_variance_power": 1.4,
    "learning_rate": 0.05,
    "max_depth": 8,
    "min_child_weight": 5,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "eval_metric": "mae",
    "seed": 42,
}


def fit(params, train_matrix, valid_matrix, rounds=3000):
    return xgb.train(
        params,
        train_matrix,
        num_boost_round=rounds,
        evals=[(valid_matrix, "valid")],
        early_stopping_rounds=100,
        verbose_eval=False,
    )


def predict(model, matrix):
    return model.predict(matrix, iteration_range=(0, model.best_iteration + 1))

In [17]:
segments = {
    "all hours": pl.lit(True),
    "workday": pl.col("is_workday"),
    "weekend / holiday": ~pl.col("is_workday"),
    "AM peak (workday 6-9)": pl.col("is_workday") & pl.col("hour").is_between(6, 9),
    "PM peak (workday 15-18)": pl.col("is_workday") & pl.col("hour").is_between(15, 18),
}


def wape_by_segment(frame, prediction_column):
    result = {}
    for segment, condition in segments.items():
        result[segment] = wape(frame.filter(condition), prediction_column)
    return result


results = []
results.append({"model": "baseline: last week", **wape_by_segment(valid_frame, "lag_1w")})
results.append({"model": "baseline: mean of 4 weeks", **wape_by_segment(valid_frame, "lag_4w_mean")})

variant_models = {}
for variant in ["all", "weighted", "post_covid"]:
    with stage(f"08.variant_{variant}", gpu=True, variant=variant):
        started = time.perf_counter()
        train_matrix, valid_matrix, rows = build_matrices(variant)
        model = fit(base_params, train_matrix, valid_matrix)
        seconds = time.perf_counter() - started

    prediction = predict(model, valid_matrix)
    scored = valid_frame.with_columns(pl.Series("prediction", prediction))
    results.append({"model": f"XGBoost: {variant}", **wape_by_segment(scored, "prediction")})
    variant_models[variant] = model
    print(f"{variant:>10}: {rows:,} train rows, best iteration {model.best_iteration}, {seconds:.1f}s")

comparison = pd.DataFrame(results).set_index("model")
comparison.round(2)

[perf] 08.variant_all: 31.41s | 1.85 cores | peak RAM 4.254 GB | CPU 10.7% | GPU mem 1382 MB
       all: 4,537,416 train rows, best iteration 286, 31.4s


[perf] 08.variant_weighted: 17.79s | 2.45 cores | peak RAM 5.284 GB | CPU 13.6% | GPU mem 1678 MB
  weighted: 4,537,416 train rows, best iteration 94, 17.8s


[perf] 08.variant_post_covid: 5.89s | 2.1 cores | peak RAM 3.953 GB | CPU 12.1% | GPU mem 1116 MB
post_covid: 911,424 train rows, best iteration 73, 5.9s


,all hours,workday,weekend / holiday,AM peak (workday 6-9),PM peak (workday 15-18)
model,,,,,
baseline: last week,20.17,16.21,37.22,15.15,13.69
baseline: mean of 4 weeks,17.54,13.93,33.07,13.09,11.81
XGBoost: all,14.09,12.06,22.79,9.99,10.11
XGBoost: weighted,14.45,12.37,23.38,10.37,10.64
XGBoost: post_covid,16.45,14.42,25.17,11.66,13.99


In [18]:
segment_names = list(segments)
model_colors = {
    "baseline: last week": viz.CONTEXT,
    "baseline: mean of 4 weeks": viz.MUTED,
    "XGBoost: all": viz.SERIES[2],
    "XGBoost: weighted": viz.ACCENT,
    "XGBoost: post_covid": viz.CONTRAST,
}

fig = go.Figure()
for model_name, color in model_colors.items():
    values = []
    for segment in segment_names:
        values.append(comparison.loc[model_name, segment])
    fig.add_trace(go.Bar(
        name=model_name,
        x=segment_names,
        y=values,
        marker_color=color,
        hovertemplate="%{x}: WAPE %{y:.1f}%<extra>%{fullData.name}</extra>",
    ))

fig.update_layout(barmode="group", hovermode="closest", legend=dict(font=dict(size=11)))
fig.update_yaxes(title_text="WAPE on valid (lower is better)", ticksuffix="%")

best_variant = comparison.loc[["XGBoost: all", "XGBoost: weighted", "XGBoost: post_covid"], "all hours"].idxmin()
best_wape = comparison.loc[best_variant, "all hours"]
baseline_wape = comparison.loc["baseline: mean of 4 weeks", "all hours"]
title = f"First try: {best_variant} reaches WAPE {best_wape:.1f}% vs {baseline_wape:.1f}% for the best baseline"
subtitle = "Station-hour entries, Jul-Dec 2022 (validation), same parameters for the three training windows"
viz.titled(fig, title, subtitle)
viz.save(fig, "08_first_models")
fig

## 6. ทดลอง Optuna 10 trial

ค้นหาพร้อมกันทั้ง hyperparameter และแบบของข้อมูล train
- max_depth, learning_rate, min_child_weight, subsample, colsample_bytree, lambda, alpha
- tweedie_variance_power ในช่วง 1.1-1.9 (EDA ประมาณไว้ ~1.4)
- แบบข้อมูล train (all / weighted / post_covid) และ half-life ถ้าเป็น weighted

ใช้ seed คงที่ (TPE sampler seed 42) รันซ้ำได้ผลเดิม
ข้อควรระวัง: เลือกโมเดลจาก valid แล้ววัดผลบน valid จะดูดีกว่าความจริงเล็กน้อย ผลสุดท้ายต้องวัดบน test ใน Step 13

matrix ของแต่ละแบบข้อมูลสร้างครั้งเดียวแล้วใช้ซ้ำ (weighted สร้างใหม่เมื่อ half-life เปลี่ยน)

In [19]:
matrix_cache = {}


def cached_matrices(variant, half_life_years):
    if variant == "weighted":
        key = (variant, round(half_life_years, 1))
    else:
        key = (variant, None)
    if key not in matrix_cache:
        train_matrix, valid_matrix, rows = build_matrices(variant, half_life_years)
        matrix_cache[key] = (train_matrix, valid_matrix)
    return matrix_cache[key]


trial_log = []


def objective(trial):
    variant = trial.suggest_categorical("variant", ["all", "weighted", "post_covid"])
    half_life = 2.0
    if variant == "weighted":
        half_life = trial.suggest_float("half_life_years", 0.5, 4.0, step=0.5)

    params = dict(base_params)
    params["max_depth"] = trial.suggest_int("max_depth", 6, 12)
    params["learning_rate"] = trial.suggest_float("learning_rate", 0.02, 0.15, log=True)
    params["min_child_weight"] = trial.suggest_float("min_child_weight", 1.0, 50.0, log=True)
    params["subsample"] = trial.suggest_float("subsample", 0.6, 1.0)
    params["colsample_bytree"] = trial.suggest_float("colsample_bytree", 0.5, 1.0)
    params["lambda"] = trial.suggest_float("lambda", 0.1, 20.0, log=True)
    params["alpha"] = trial.suggest_float("alpha", 0.01, 5.0, log=True)
    params["tweedie_variance_power"] = trial.suggest_float("tweedie_variance_power", 1.1, 1.9)

    started = time.perf_counter()
    train_matrix, valid_matrix = cached_matrices(variant, half_life)
    model = fit(params, train_matrix, valid_matrix)
    prediction = predict(model, valid_matrix)
    error = float(np.abs(y_valid - prediction).sum() / y_valid.sum() * 100)
    seconds = time.perf_counter() - started

    trial_log.append({"trial": trial.number, "variant": variant, "wape": error,
                      "best_iteration": model.best_iteration, "seconds": seconds})
    print(f"trial {trial.number:>2}: {variant:>10}  WAPE {error:.2f}%  iterations {model.best_iteration:>4}  {seconds:.1f}s", flush=True)
    return error


optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42))

with stage("08.optuna_10_trials", gpu=True, trials=10):
    study.optimize(objective, n_trials=10)

print()
print(f"best trial: {study.best_trial.number}, WAPE {study.best_value:.2f}%")
study.best_params

trial  0:   weighted  WAPE 14.89%  iterations  212  22.2s


trial  1:        all  WAPE 13.97%  iterations  440  35.9s


trial  2: post_covid  WAPE 16.54%  iterations   94  4.6s


trial  3:        all  WAPE 14.23%  iterations  606  33.6s


trial  4:        all  WAPE 13.75%  iterations  803  69.3s


...

trial  7:        all  WAPE 14.41%  iterations  558  31.4s


trial  8:   weighted  WAPE 14.50%  iterations  131  23.4s


trial  9:   weighted  WAPE 13.94%  iterations  370  68.2s


[perf] 08.optuna_10_trials: 319.41s | 1.52 cores | peak RAM 8.648 GB | CPU 8.7% | GPU mem 2836 MB

best trial: 4, WAPE 13.75%


{'variant': 'all',
 'max_depth': 9,
 'learning_rate': 0.02902611884153732,
 'min_child_weight': 44.39102767051397,
 'subsample': 0.9100531293444458,
 'colsample_bytree': 0.9697494707820946,
 'lambda': 11.455777613940603,
 'alpha': 0.41087915453240803,
 'tweedie_variance_power': 1.8374993880184936}

In [20]:
trials = pd.DataFrame(trial_log)
seconds_per_trial = trials["seconds"].mean()

print(f"average time per trial: {seconds_per_trial:.1f}s")
print(f"estimated time for 50 trials : {50 * seconds_per_trial / 60:.0f} min")
print(f"estimated time for 100 trials: {100 * seconds_per_trial / 60:.0f} min")
trials.round(2)

average time per trial: 31.9s
estimated time for 50 trials : 27 min
estimated time for 100 trials: 53 min


,trial,variant,wape,best_iteration,seconds
0,0,weighted,14.89,212,22.16
1,1,all,13.97,440,35.89
2,2,post_covid,16.54,94,4.61
3,3,all,14.23,606,33.60
4,4,all,13.75,803,69.26
5,5,weighted,14.93,135,19.92
6,6,weighted,15.25,48,10.94
7,7,all,14.41,558,31.36
8,8,weighted,14.50,131,23.44
9,9,weighted,13.94,370,68.19


## สรุปส่วนที่ 2

- pipeline ทำงานครบบน GPU แต่ละโมเดลใช้เวลา 3-30 วินาที ใช้ VRAM สูงสุด ~2.8 GB จาก 8 GB
- รอบแรกด้วย parameter ตั้งต้น ใช้ข้อมูลทั้งหมดตั้งแต่ปี 2011 ได้ WAPE 14.1% ชนะ baseline ค่าเฉลี่ย 4 สัปดาห์ (17.5%) ทุก segment
- ส่วนที่ชนะมากที่สุดคือวันหยุด (33.1% เหลือ 22.8%) เพราะโมเดลมี feature วันหยุดและรู้ว่าสัปดาห์ก่อนเป็นวันหยุดหรือไม่ ซึ่ง baseline ไม่รู้
- ใช้ข้อมูลทั้งหมดดีกว่าใช้แค่หลัง COVID (16.5%) แม้ระดับจะต่างกันมาก เพราะ lag และ feature ระดับล่าสุดบอกระดับปัจจุบันให้แล้ว ข้อมูลเก่ายังช่วยสอนรูปแบบของชั่วโมง วัน และวันหยุด
- Optuna 10 trial ได้ดีที่สุด 13.75% ใช้เวลาเฉลี่ย ~15 วินาทีต่อ trial (ตัวเลขนี้วัดบน valid ที่ใช้เลือกโมเดลด้วย จึงดูดีกว่าความจริงเล็กน้อย)

ต่อไป: ส่วนที่ 3 จูนเต็ม (ต้องยืนยันจำนวน trial และเวลาก่อน) → fit สุดท้าย → quantile P10/P50/P90 → SHAP → ประเมินบน test

ต่อไป: ส่วนที่ 3 จูนเต็ม (ต้องยืนยันจำนวน trial และเวลาก่อน) → fit สุดท้าย → quantile P10/P50/P90 → SHAP → ประเมินบน test

# ส่วนที่ 3: จูนเต็ม, quantile, SHAP และประเมินบน test

## 7. Optuna 50 trial

ใช้ search space และ objective เดียวกับส่วนที่ 2
ผลของแต่ละ trial เก็บใน SQLite (`outputs/models/m1_optuna.db`) ถ้ารัน notebook ซ้ำจะรันต่อเฉพาะ trial ที่ยังขาด ไม่ต้องจูนใหม่ทั้งหมด

ตอนทดสอบใช้เวลารวม ~27 นาที trial ช่วงแรกใช้ ~15 วินาที แต่ช่วงหลังใช้ 40-70 วินาที เพราะ Optuna เจอว่าต้นไม้ลึกและ learning rate ต่ำให้ผลดีกว่า จึงเลือกค่าแบบนั้นบ่อยขึ้น

กันค้าง
- แต่ละ trial มีเพดาน 3,000 รอบและ early stopping 100 รอบ จึงจบเองเสมอ
- ทั้งรอบจูนมีเพดานเวลา 60 นาที (`timeout`) ถ้าเกินจะหยุดแล้วใช้ trial ที่ดีที่สุดที่ได้ ผลยังบันทึกใน SQLite รันต่อทีหลังได้
- พิมพ์ความคืบหน้าทุก trial พร้อมเวลาที่ใช้ไปและเวลาที่น่าจะเหลือ

In [21]:
import json

from bart.config import MODELS, TAB

TRIALS = 50
storage = f"sqlite:///{MODELS / 'm1_optuna.db'}"

study = optuna.create_study(
    study_name="m1_forecast",
    storage=storage,
    load_if_exists=True,
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=42),
)

finished = 0
for trial in study.trials:
    if trial.state == optuna.trial.TrialState.COMPLETE:
        finished += 1
remaining = TRIALS - finished
print(f"finished trials: {finished}, running {max(remaining, 0)} more")

tuning_started = time.perf_counter()


def report_progress(study, trial):
    done = 0
    for finished_trial in study.trials:
        if finished_trial.state == optuna.trial.TrialState.COMPLETE:
            done += 1
    elapsed = time.perf_counter() - tuning_started
    finished_now = done - finished
    seconds_each = elapsed / max(finished_now, 1)
    left = (TRIALS - done) * seconds_each
    print(f"    {done}/{TRIALS} done, {elapsed / 60:.1f} min so far, about {left / 60:.0f} min left, best {study.best_value:.2f}%", flush=True)


if remaining > 0:
    with stage("08.optuna_full", gpu=True, trials=remaining):
        study.optimize(objective, n_trials=remaining, timeout=60 * 60, callbacks=[report_progress])

print()
print(f"best trial: {study.best_trial.number}, valid WAPE {study.best_value:.2f}%")
study.best_params

finished trials: 50, running 0 more

best trial: 41, valid WAPE 13.50%


{'variant': 'all',
 'max_depth': 12,
 'learning_rate': 0.024697759657074526,
 'min_child_weight': 13.338470356857728,
 'subsample': 0.8101114780604701,
 'colsample_bytree': 0.8182291712098407,
 'lambda': 5.688816629954367,
 'alpha': 2.1297450580353896,
 'tweedie_variance_power': 1.6192401160029408}

In [22]:
history_rows = []
best_so_far = None
for trial in study.trials:
    value = trial.value
    if trial.state != optuna.trial.TrialState.COMPLETE or value is None:
        continue
    if best_so_far is None or value < best_so_far:
        best_so_far = value
    history_rows.append({
        "trial": trial.number,
        "wape": value,
        "best_so_far": best_so_far,
        "variant": trial.params["variant"],
    })
history = pd.DataFrame(history_rows)
history.to_csv(TAB / "m1_optuna_trials.csv", index=False)

variant_colors = {"all": viz.SERIES[2], "weighted": viz.ACCENT, "post_covid": viz.CONTRAST}

fig = go.Figure()
for variant, color in variant_colors.items():
    rows = history[history["variant"] == variant]
    fig.add_trace(go.Scatter(
        x=rows["trial"], y=rows["wape"], mode="markers", name=f"trial ({variant})",
        marker=dict(color=color, size=9, opacity=0.8),
        hovertemplate="trial %{x}: WAPE %{y:.2f}%<extra>" + variant + "</extra>",
    ))
fig.add_trace(go.Scatter(
    x=history["trial"], y=history["best_so_far"], mode="lines", name="best so far",
    line=dict(color=viz.INK, width=2, shape="hv"),
    hovertemplate="after trial %{x}: best %{y:.2f}%<extra></extra>",
))
fig.update_layout(hovermode="closest")
fig.update_xaxes(title_text="trial")
fig.update_yaxes(title_text="valid WAPE", ticksuffix="%")

title = f"Optuna: best valid WAPE {study.best_value:.2f}% after {len(history)} trials"
subtitle = "Each dot is one trial (colour = training window). The line is the best result so far"
viz.titled(fig, title, subtitle)
viz.save(fig, "08_optuna_history")
fig

## 8. โมเดลสุดท้าย

1. เทรนด้วย parameter ที่ดีที่สุดบน train โดยใช้ valid ทำ early stopping เพื่อหาจำนวนรอบ
   ลองทั้ง learning rate ที่จูนได้ และครึ่งหนึ่งของค่านั้น (เทรนนานขึ้นแต่มักแม่นกว่า) แล้วเลือกตัวที่ valid ดีกว่า
   ตอนทดสอบ ลด learning rate ลงครึ่งหนึ่งดีขึ้นแค่ ~0.01 จุดบน valid (เท่ากันในทางปฏิบัติ) ส่วนการเพิ่มความลึกของต้นไม้เป็น 14 ชั้นดีขึ้น 0.02 จุดแต่ช้าลงกว่า 2 เท่า ผลที่ได้จึงใกล้ขีดจำกัดของ feature ชุดนี้แล้ว
   เลือกจาก valid เท่านั้น ไม่ใช้ test เลือก
2. เทรนใหม่บน train + valid ด้วยจำนวนรอบเท่าเดิมบวก 5% (ข้อมูลมากขึ้นเล็กน้อย) จะได้ใช้ข้อมูลล่าสุดก่อนช่วง test
3. ใช้วิธีเดียวกันกับโมเดล quantile (P10, P50, P90) ซึ่งใช้ objective `reg:quantileerror` เทรนครั้งเดียวได้ทั้ง 3 quantile
   โมเดล quantile ใช้ต้นไม้ไม่เกิน 8 ชั้นและ learning rate 0.05 เพราะใช้แค่บอกช่วงความไม่แน่นอน ถ้าใช้ค่าที่จูนได้ (12 ชั้น, 0.025) กับ 3 quantile พร้อมกันจะช้ามาก (ตอนทดสอบเกิน 15 นาที)

test (ม.ค. - 2 พ.ค. 2023) ใช้ครั้งเดียวตรงนี้

In [23]:
best = study.best_params
best_variant = best["variant"]
best_half_life = best.get("half_life_years", 2.0)

final_params = dict(base_params)
for name in ["max_depth", "learning_rate", "min_child_weight", "subsample", "colsample_bytree",
             "lambda", "alpha", "tweedie_variance_power"]:
    final_params[name] = best[name]

quantile_params = dict(final_params)
del quantile_params["tweedie_variance_power"]
quantile_params["max_depth"] = min(final_params["max_depth"], 8)
quantile_params["learning_rate"] = 0.05
quantile_params["objective"] = "reg:quantileerror"
quantile_params["quantile_alpha"] = np.array([0.1, 0.5, 0.9])
quantile_params["eval_metric"] = "quantile"

print("training window:", best_variant)
pd.Series(final_params).to_frame("value")

training window: all


,value
device,cuda
tree_method,hist
objective,reg:tweedie
tweedie_variance_power,1.61924
learning_rate,0.024698
max_depth,12
min_child_weight,13.33847
subsample,0.810111
colsample_bytree,0.818229
eval_metric,mae


In [24]:
def valid_wape(model, matrix):
    prediction = predict(model, matrix)
    return float(np.abs(y_valid - prediction).sum() / y_valid.sum() * 100)


train_matrix, valid_matrix = cached_matrices(best_variant, best_half_life)

candidates = {}
rate_rows = []
with stage("08.learning_rate_check", gpu=True):
    for label, rate in [("tuned", final_params["learning_rate"]), ("half", final_params["learning_rate"] / 2)]:
        params = dict(final_params)
        params["learning_rate"] = rate
        started = time.perf_counter()
        model = fit(params, train_matrix, valid_matrix, rounds=6000)
        error = valid_wape(model, valid_matrix)
        candidates[label] = (params, model, error)
        rate_rows.append({
            "learning_rate": label,
            "value": round(rate, 4),
            "valid_wape": round(error, 3),
            "trees": model.best_iteration + 1,
            "seconds": round(time.perf_counter() - started),
        })

chosen = "tuned"
if candidates["half"][2] < candidates["tuned"][2]:
    chosen = "half"
final_params, point_check, chosen_wape = candidates[chosen]

print(f"chosen: {chosen} learning rate ({final_params['learning_rate']:.4f}), valid WAPE {chosen_wape:.3f}%")
pd.DataFrame(rate_rows)

[perf] 08.learning_rate_check: 337.43s | 1.0 cores | peak RAM 8.723 GB | CPU 6.2% | GPU mem 3392 MB
chosen: half learning rate (0.0123), valid WAPE 13.487%


,learning_rate,value,valid_wape,trees,seconds
0,tuned,0.0247,13.501,614,137
1,half,0.0123,13.487,926,201


In [25]:
test_frame = feature_table.filter(pl.col("split") == "test").with_columns(is_workday().alias("is_workday"))
full_frame = pl.concat([training_rows(best_variant), feature_table.filter(pl.col("split") == "valid")])

X_test = to_model_input(test_frame)
y_test = test_frame["target"].to_numpy()

test_start = date(2023, 1, 1)
if best_variant == "weighted":
    days_back = (pl.lit(test_start) - pl.col("date")).dt.total_days()
    full_weight = full_frame.select((0.5 ** (days_back / 365.25 / best_half_life)).alias("w"))["w"].to_numpy()
else:
    full_weight = None

with stage("08.final_models", gpu=True):
    quantile_check = fit(quantile_params, train_matrix, valid_matrix, rounds=1500)

    point_rounds = int((point_check.best_iteration + 1) * 1.05)
    quantile_rounds = int((quantile_check.best_iteration + 1) * 1.05)

    full_matrix = xgb.QuantileDMatrix(to_model_input(full_frame), full_frame["target"].to_numpy(),
                                      weight=full_weight, enable_categorical=True, max_bin=256)
    test_matrix = xgb.QuantileDMatrix(X_test, y_test, ref=full_matrix, enable_categorical=True, max_bin=256)

    point_model = xgb.train(final_params, full_matrix, num_boost_round=point_rounds)
    quantile_model = xgb.train(quantile_params, full_matrix, num_boost_round=quantile_rounds)

print(f"point model   : {point_rounds} rounds on {full_frame.height:,} rows")
print(f"quantile model: {quantile_rounds} rounds")

point_model.save_model(MODELS / "m1_point.json")
quantile_model.save_model(MODELS / "m1_quantile.json")
with open(MODELS / "m1_best_params.json", "w") as handle:
    json.dump({"tuned": best, "variant": best_variant, "final_learning_rate": final_params["learning_rate"],
               "point_rounds": point_rounds, "quantile_rounds": quantile_rounds}, handle, indent=2)

[perf] 08.final_models: 784.01s | 1.03 cores | peak RAM 10.236 GB | CPU 6.3% | GPU mem 3523 MB
point model   : 972 rounds on 4,758,216 rows
quantile model: 1575 rounds


In [26]:
point_prediction = point_model.predict(test_matrix)
quantile_prediction = np.sort(quantile_model.predict(test_matrix), axis=1)

scored = test_frame.with_columns(
    pl.Series("prediction", point_prediction),
    pl.Series("p10", quantile_prediction[:, 0]),
    pl.Series("p50", quantile_prediction[:, 1]),
    pl.Series("p90", quantile_prediction[:, 2]),
)

crossing_before_sort = 0
raw_quantiles = quantile_model.predict(test_matrix)
for low, middle, high in raw_quantiles[:50000]:
    if not (low <= middle <= high):
        crossing_before_sort += 1
print(f"quantile crossings in the first 50,000 test rows before sorting: {crossing_before_sort}")

quantile crossings in the first 50,000 test rows before sorting: 0


## 9. ผลบน test

ตัวชี้วัด
- WAPE (ตัวหลัก) = ผลรวม |จริง - พยากรณ์| / ผลรวมค่าจริง
- MAE, RMSE (หน่วยเป็นคนต่อสถานี-ชั่วโมง), R²
- coverage = สัดส่วนค่าจริงที่อยู่ในช่วง P10-P90 (เป้าคือ ~80%)

In [27]:
def metrics(frame, prediction_column):
    rows = frame.filter(pl.col(prediction_column).is_not_null())
    actual = rows["target"].to_numpy()
    predicted = rows[prediction_column].to_numpy()
    errors = actual - predicted
    total_variation = ((actual - actual.mean()) ** 2).sum()
    return {
        "WAPE %": 100 * np.abs(errors).sum() / actual.sum(),
        "MAE": np.abs(errors).mean(),
        "RMSE": np.sqrt((errors ** 2).mean()),
        "R2": 1 - (errors ** 2).sum() / total_variation,
    }


def coverage(frame):
    inside = frame.filter((pl.col("target") >= pl.col("p10")) & (pl.col("target") <= pl.col("p90"))).height
    return 100 * inside / frame.height


test_rows = []
test_rows.append({"model": "baseline: last week", **metrics(scored, "lag_1w")})
test_rows.append({"model": "baseline: mean of 4 weeks", **metrics(scored, "lag_4w_mean")})
test_rows.append({"model": "M1 point (Tweedie)", **metrics(scored, "prediction")})
test_rows.append({"model": "M1 quantile P50", **metrics(scored, "p50")})

test_metrics = pd.DataFrame(test_rows).set_index("model")
test_metrics.to_csv(TAB / "m1_test_metrics.csv")

print(f"P10-P90 coverage on test: {coverage(scored):.1f}% (target ~80%)")
test_metrics.round(3)

P10-P90 coverage on test: 79.1% (target ~80%)


,WAPE %,MAE,RMSE,R2
model,,,,
baseline: last week,17.953,18.948,53.618,0.934
baseline: mean of 4 weeks,17.116,18.065,50.929,0.940
M1 point (Tweedie),14.499,15.303,44.406,0.955
M1 quantile P50,14.479,15.281,42.986,0.958


In [28]:
segment_rows = []
for model_name, column in [("baseline: mean of 4 weeks", "lag_4w_mean"), ("M1 point (Tweedie)", "prediction")]:
    values = {}
    for segment, condition in segments.items():
        values[segment] = wape(scored.filter(condition), column)
    segment_rows.append({"model": model_name, **values})

coverage_values = {}
for segment, condition in segments.items():
    coverage_values[segment] = coverage(scored.filter(condition))
segment_rows.append({"model": "P10-P90 coverage %", **coverage_values})

by_segment = pd.DataFrame(segment_rows).set_index("model")
by_segment.round(1)

,all hours,workday,weekend / holiday,AM peak (workday 6-9),PM peak (workday 15-18)
model,,,,,
baseline: mean of 4 weeks,17.1,15.5,24.6,14.9,13.9
M1 point (Tweedie),14.5,13.6,19.0,12.2,12.3
P10-P90 coverage %,79.1,77.7,82.1,68.0,72.5


In [29]:
station_day = (
    scored
    .group_by("station", "date")
    .agg(pl.col("target").sum(), pl.col("prediction").sum(), pl.col("lag_4w_mean").sum())
)
station_day_model = wape(station_day, "prediction")
station_day_baseline = wape(station_day, "lag_4w_mean")

print(f"station-day WAPE: model {station_day_model:.2f}% vs baseline {station_day_baseline:.2f}%")

station-day WAPE: model 8.78% vs baseline 11.92%


In [30]:
fig = go.Figure()
segment_names = list(segments)
for model_name, color in [("baseline: mean of 4 weeks", viz.CONTEXT), ("M1 point (Tweedie)", viz.ACCENT)]:
    values = []
    labels = []
    for segment in segment_names:
        value = by_segment.loc[model_name, segment]
        values.append(value)
        labels.append(f"{value:.1f}%")
    fig.add_trace(go.Bar(
        name=model_name, x=segment_names, y=values, marker_color=color,
        text=labels, textposition="outside",
        hovertemplate="%{x}: WAPE %{y:.1f}%<extra>%{fullData.name}</extra>",
    ))

fig.update_layout(barmode="group", hovermode="closest")
fig.update_yaxes(title_text="WAPE on test (lower is better)", ticksuffix="%", rangemode="tozero")

model_wape = test_metrics.loc["M1 point (Tweedie)", "WAPE %"]
baseline_wape = test_metrics.loc["baseline: mean of 4 weeks", "WAPE %"]
improvement = 100 * (baseline_wape - model_wape) / baseline_wape
title = f"On unseen 2023 data, M1 cuts the error by {improvement:.0f}% (WAPE {model_wape:.1f}% vs {baseline_wape:.1f}%)"
subtitle = "Station-hour entries, Jan 1 - May 2 2023 (test set, used once), M1 vs the best baseline"
viz.titled(fig, title, subtitle)
viz.save(fig, "08_test_by_segment")
fig

ตัวอย่างการใช้งาน: สัปดาห์หนึ่งของ test ที่สถานี EMBR ค่าจริงเทียบกับ P50 และช่วง P10-P90

In [31]:
example = scored.filter(
    (pl.col("station") == "EMBR") & pl.col("date").is_between(date(2023, 3, 6), date(2023, 3, 12))
).sort("date", "hour").with_columns(
    pl.col("date").cast(pl.Datetime) + pl.duration(hours=pl.col("hour"))
).rename({"date": "time"})

band_top = go.Scatter(x=example["time"], y=example["p90"], mode="lines", line=dict(width=0),
                      showlegend=False, hoverinfo="skip")
band = go.Scatter(x=example["time"], y=example["p10"], mode="lines", line=dict(width=0), fill="tonexty",
                  fillcolor="rgba(42, 120, 214, 0.18)", name="P10-P90", hoverinfo="skip")
p50_line = go.Scatter(x=example["time"], y=example["p50"], mode="lines", name="P50",
                      line=dict(color=viz.ACCENT, width=2),
                      hovertemplate="%{x|%a %H:00}: P50 %{y:,.0f}<extra></extra>")
actual_line = go.Scatter(x=example["time"], y=example["target"], mode="lines", name="actual",
                         line=dict(color=viz.INK, width=2, dash="dot"),
                         hovertemplate="%{x|%a %H:00}: actual %{y:,.0f}<extra></extra>")

fig = go.Figure([band_top, band, p50_line, actual_line])
fig.update_yaxes(title_text="entries per hour", tickformat="~s")

inside = example.filter((pl.col("target") >= pl.col("p10")) & (pl.col("target") <= pl.col("p90"))).height
title = f"EMBR, week of 6 March 2023: {100 * inside / example.height:.0f}% of hours fall inside the P10-P90 band"
subtitle = "Hourly entries, actual vs forecast made 7 days ahead"
viz.titled(fig, title, subtitle)
viz.save(fig, "08_example_week")
fig

## 10. อะไรขับเคลื่อนการพยากรณ์ (SHAP)

ใช้ TreeSHAP ที่มีใน XGBoost คำนวณบน GPU (`pred_contribs=True`) ให้กับ 20,000 แถวที่สุ่มจาก test
ค่า mean |SHAP| บอกว่าแต่ละ feature เปลี่ยนค่าพยากรณ์โดยเฉลี่ยมากแค่ไหน (หน่วยเป็นค่าก่อนแปลงกลับของ Tweedie คือ log ของค่าพยากรณ์)

In [32]:
shap_sample = test_frame.sample(n=20000, seed=11)
shap_matrix = xgb.DMatrix(to_model_input(shap_sample), enable_categorical=True)

with stage("08.shap", gpu=True):
    contributions = point_model.predict(shap_matrix, pred_contribs=True)

mean_abs = np.abs(contributions[:, :-1]).mean(axis=0)
importance_rows = []
for name, value in zip(feature_columns, mean_abs):
    importance_rows.append({"feature": name, "mean_abs_shap": float(value)})
importance = pd.DataFrame(importance_rows).sort_values("mean_abs_shap", ascending=False)
importance.to_csv(TAB / "m1_shap_importance.csv", index=False)

top = importance.head(15).iloc[::-1]
fig = go.Figure(go.Bar(
    x=top["mean_abs_shap"], y=top["feature"], orientation="h", marker_color=viz.ACCENT,
    hovertemplate="%{y}: %{x:.3f}<extra></extra>",
))
fig.update_layout(hovermode="closest", showlegend=False, margin=dict(l=170))
fig.update_xaxes(title_text="mean |SHAP| (log scale of the forecast)", showgrid=True, gridcolor=viz.GRID)
fig.update_yaxes(showgrid=False)

title = "The forecast leans mostly on the same hour over the last 4 weeks"
subtitle = "Top 15 features by mean absolute SHAP value, 20,000 random test rows"
viz.titled(fig, title, subtitle)
viz.save(fig, "08_shap_importance", height=560)
fig

[perf] 08.shap: 293.51s | 1.0 cores | peak RAM 9.152 GB | CPU 6.9% | GPU mem 3763 MB


## สรุป: Insight → Pain point

pain point ของ BART ในข้อนี้คือ ทุกสัปดาห์ต้องกำหนดความยาวขบวน ความถี่ และพนักงานของสัปดาห์หน้า ถ้าพยากรณ์ผิดจะได้รถเกิน (เสียต้นทุน) หรือรถแน่น (เสียบริการ)

| # | Insight (ตัวเลข) | Pain point | ตอบโจทย์แค่ไหน | ข้อเสนอ |
|---|---|---|---|---|
| 1 | บน test (ม.ค.-2 พ.ค. 2023 ข้อมูลที่ไม่เคยเห็น) M1 ได้ WAPE 14.5% เทียบกับ baseline ค่าเฉลี่ย 4 สัปดาห์ 17.1% ลด error ลง ~15% และ R² 0.955 | ความแม่นของแผนรายสัปดาห์ | ตรงจุด | ใช้ M1 P50 แทนค่าเฉลี่ยย้อนหลังในการวางแผนรายสัปดาห์ |
| 2 | ระดับสถานี-วัน (ใช้วางพนักงานรายวัน) WAPE 8.8% เทียบกับ 11.9% ลด error ลง ~26% | การวางพนักงานรายวัน | ตรงจุด | ใช้ผลรวมรายวันของ M1 วางกำลังพนักงานสถานี |
| 3 | ชนะ baseline มากที่สุดในวันหยุด/สุดสัปดาห์ (19.0% เทียบกับ 24.6%) ช่วง peak เช้า 12.2% เทียบกับ 14.9% และ peak เย็น 12.3% เทียบกับ 13.9% | วันที่ตารางเดินรถพลาดบ่อย | ตรงจุด | ใช้ M1 กับสัปดาห์ที่มีวันหยุดเป็นพิเศษ เพราะเป็นจุดที่วิธีเดิมพลาดมากที่สุด |
| 4 | ช่วง P10-P90 ครอบค่าจริง 79.1% ใกล้เป้า 80% แต่ช่วง peak แคบเกินไป (เช้า 68%, เย็น 72.5%) ตัวอย่าง EMBR สัปดาห์ 6 มี.ค. 2023 ชั่วโมง 17:00 วันอังคารค่าจริงสูงกว่าพยากรณ์ 24% วันพุธ 16% แต่วันพฤหัสต่ำกว่า 10% | ขบวนสำรอง (P90) | บางส่วน ใช้ได้ทั้งวัน แต่ช่วง peak ยังกันพลาดไม่พอ | ช่วง peak ยังไม่ควรใช้ P90 ตรงๆ เป็นขบวนสำรอง ควรปรับช่วงให้กว้างขึ้นโดยใช้ error ของ valid (calibration) ก่อนใช้งานจริง |
| 5 | SHAP: ค่าพยากรณ์อิงค่ามัธยฐาน/ค่าเฉลี่ย 4 สัปดาห์ของชั่วโมงเดียวกันมากที่สุด รองลงมาคือค่าเมื่อปีก่อน ชั่วโมง สถานี และระดับทั้งระบบ | ความน่าเชื่อถือของโมเดล | ตรงจุด อธิบายได้และตรงกับ EDA | ผู้ใช้เข้าใจได้ง่ายว่าโมเดลคือ "ค่าเฉลี่ยล่าสุดที่ปรับตามปฏิทินและระดับของระบบ" |
| 6 | จูน 50 trial ผลคงที่ตั้งแต่ ~trial 26 (13.5% บน valid) ใช้ข้อมูลทั้งหมดตั้งแต่ปี 2011 ดีที่สุด ลองต้นไม้ลึกขึ้นหรือ learning rate ต่ำลงแล้วดีขึ้นไม่ถึง 0.05 จุด | ต้นทุนการดูแลโมเดล | ตรงจุด | เทรนใหม่ทุกไตรมาสด้วย parameter เดิมได้ ไม่ต้องจูนใหม่ทุกครั้ง (ตรวจใน M3) |

ทรัพยากรที่ใช้: สร้าง feature 5 ล้านแถวด้วย Polars ~2 วินาที, จูน 50 trial บน RTX 4060 ~28 นาที, เทียบ learning rate ~2.6 นาที, เทรนโมเดลสุดท้าย ~6.5 นาที, SHAP 20,000 แถว ~1.8 นาที, VRAM สูงสุด ~5.6 GB

ข้อจำกัด
- test มีแค่ 4 เดือนแรกของปี 2023 ยังไม่ได้ทดสอบช่วงฤดูร้อนหรือวันหยุดยาว (M3 จะทดสอบแบบ rolling หลายไตรมาส)
- โมเดล quantile ใช้ต้นไม้ตื้นกว่าโมเดลหลักเพื่อให้เทรนเร็ว และเทรนจนถึงขีดจำกัด 1,500 รอบ ช่วงอาจดีขึ้นได้ถ้าเทรนนานกว่านี้
- พยากรณ์ entries อย่างเดียว ไม่ได้พยากรณ์ exits หรือโหลดบนขบวน

ต่อไป: 09_m2_station_typology